# IO Cloud Agent Cloud — LLM + MCP Agent Tutorial

## Core Idea

In the previous tutorial we called MCP tools manually. But the real usage of Agent Cloud is:

```
User -> Natural Language -> LLM (GLM-5.1) -> Auto-select Tool -> MCP Server -> Return Result -> LLM Summary -> User
```

You just say "show me the available GPUs", and the LLM automatically decides which API to call, what parameters to pass, and then tells you the result in plain language.

**This tutorial uses:**
- LLM: **GLM-5.1** (called via IO Intelligence API)
- MCP Server: **IO Cloud** (io.net GPU infrastructure management)
- Both services are provided by io.net, using different API Keys

## 0. Environment Setup

In [ ]:
import os
# os.environ['http_proxy']  = 'http://127.0.0.1:7890'
# os.environ['https_proxy'] = 'http://127.0.0.1:7890'

In [ ]:
import json, asyncio
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# IO Intelligence API key for calling GLM-5.1
# This is the Intelligence Key, not the Cloud/MCP key
INTELLIGENCE_KEY = 'io-v2-****'
LLM_BASE_URL = 'https://api.intelligence.io.solutions/api/v1'
LLM_MODEL = 'zai-org/GLM-5.1'

# IO Cloud MCP key for GPU infrastructure operations
# This is the Cloud Key, not the Intelligence key
CLOUD_KEY = 'io-v2-*****'
MCP_URL = 'https://mcp.io.solutions/mcp'
MCP_HEADERS = {'x-api-key': CLOUD_KEY}

llm = OpenAI(api_key=INTELLIGENCE_KEY, base_url=LLM_BASE_URL)

print('OK')

---

## 1. Auto-fetch Tool Definitions from MCP Server

Connect to the MCP server, pull the schema for all tools, then convert them to OpenAI function calling format.

This way the LLM knows what "capabilities" it has available.

In [ ]:
async def get_mcp_tools():
    """Fetch the tool list from the MCP server and convert to OpenAI tools format."""
    async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS, timeout=60) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            result = await session.list_tools()
            
            openai_tools = []
            for tool in result.tools:
                openai_tools.append({
                    'type': 'function',
                    'function': {
                        'name': tool.name,
                        'description': tool.description or '',
                        'parameters': tool.inputSchema or {'type': 'object', 'properties': {}}
                    }
                })
            return openai_tools

tools = await get_mcp_tools()
print(f'Fetched {len(tools)} tools in total:')
for t in tools:
    print(f'  {t["function"]["name"]}')

---

## 2. Build the Agent Loop

Core logic:
1. User speaks natural language -> send to GLM-5.1
2. GLM-5.1 returns tool_calls -> execute via MCP
3. Tool results are sent back to GLM-5.1 -> generate a human-readable answer

In [ ]:
async def call_mcp_tool(tool_name, arguments):
    """Execute a tool call through the MCP server."""
    async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS, timeout=60) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments=arguments)
            text = result.content[0].text if result.content else ''
            return text


async def agent_chat(user_message, tools, verbose=True):
    """
    Complete Agent loop:
    User input -> LLM reasoning -> Tool call -> LLM summary -> Return
    """
    messages = [
        {'role': 'system', 'content': (
            'You are an IO Cloud GPU infrastructure management assistant. '
            'You can query hardware, estimate prices, deploy containers, manage deployments, etc. via tools. '
            'Please answer concisely and clearly in English.'
        )},
        {'role': 'user', 'content': user_message}
    ]

    if verbose:
        print(f'[USER] {user_message}')

    # Round 1: LLM decides which tool to call
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=tools,
        max_tokens=2000
    )

    assistant_msg = resp.choices[0].message
    messages.append(assistant_msg)

    # If the LLM requests tool calls
    if assistant_msg.tool_calls:
        if verbose and assistant_msg.content:
            print(f'\n[GLM-5.1] {assistant_msg.content}')

        for tc in assistant_msg.tool_calls:
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

            if verbose:
                print(f'\n[TOOL CALL] {func_name}({json.dumps(func_args, ensure_ascii=False)})')

            # Execute the tool via MCP
            tool_result = await call_mcp_tool(func_name, func_args)

            if verbose:
                preview = tool_result[:500] + '...' if len(tool_result) > 500 else tool_result
                print(f'\n[TOOL RESULT] ({len(tool_result)} chars):\n{preview}')

            # Send tool results back to the LLM
            messages.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': tool_result
            })

        # Round 2: LLM generates answer based on tool results
        resp2 = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            max_tokens=2000
        )
        final_answer = resp2.choices[0].message.content
    else:
        final_answer = assistant_msg.content

    if verbose:
        print(f'\n[ANSWER]\n{final_answer}')

    return final_answer


print('Agent loop defined')

---

## 3. Demo: Query Hardware with Natural Language

Just ask in plain language, and GLM-5.1 will automatically call `caas_get_hardware_ids`.

In [ ]:
answer = await agent_chat('Show me the available GPU hardware. Which one is the cheapest?', tools)

---

## 4. Demo: Estimate Prices with Natural Language

Ask about pricing, and GLM-5.1 will automatically choose `caas_get_price_estimate` and fill in the parameters.

In [ ]:
answer = await agent_chat('How much does it cost to deploy an RTX 4090 for 1 hour? Use hardware_id=12, location_ids=[2]', tools)

---

## 5. Demo: View My Deployments

Ask about deployments, and GLM-5.1 will call `caas_list_deployments`.

In [ ]:
answer = await agent_chat('List all container deployments under my account', tools)

---

## 6. Demo: Complex Multi-step Query

Try a more complex question and see if GLM-5.1 can break it down on its own.

In [ ]:
answer = await agent_chat('I want to know if there are any H100 GPUs available right now, and if so, how much per hour?', tools)

---

## 7. Interactive Chat

You can type your own questions and try it out:

In [ ]:
# Change the question here, then run the cell
your_question = 'How much would it cost to deploy the cheapest GPU for 2 hours?'

answer = await agent_chat(your_question, tools)

---

## Summary

### Architecture Overview

```
User (Natural Language)
  |
  v
GLM-5.1 (IO Intelligence API)     <-- "Brain": understands intent + selects tools
  | tool_calls
  v
MCP Server (IO Cloud)              <-- "Hands": executes GPU infrastructure operations
  | results
  v
GLM-5.1                            <-- Summarizes results, answers in plain language
  |
  v
User (sees a readable answer)
```

### Two Keys and Their Roles

| Key | Service | Purpose |
|-----|---------|--------|
| Intelligence Key | IO Intelligence API | Call LLM models like GLM-5.1 |
| Cloud Key | IO Cloud MCP Server | Manage GPU hardware and container deployments |

This is the core idea of Agent Cloud: use AI models to drive infrastructure management — all you need to do is talk.